# Notebook 4 — Perbandingan model, domain shift, dan analisis kegagalan

Notebook ini **tidak melatih apa pun**. Semua angka dihitung dari prediksi yang sudah disimpan Notebook 1–3
(`runs/<model>_<fraksi>/preds_<split>.json`) dan anotasi dari Notebook 0, dengan evaluasi pycocotools yang sama.

| Bagian | Menjawab |
|---|---|
| 1. Ringkasan AP | perbandingan 3 model × 3 fraksi data latih (variabel NRP digit 3) |
| 2. Kurva belajar | apakah training sudah konvergen, kapan model terbaik dicapai |
| 3. Statistik split | seberapa beda train, valid, test, dan test_own (ukuran gambar, kecerahan, ketajaman, distribusi kelas) |
| 4. Frekuensi kelas | pengaruh jumlah contoh per kelas di train terhadap AP |
| 5. AP vs ambang IoU | kualitas tepi mask tiap arsitektur |
| 6. Ukuran objek | AP objek kecil, sedang, besar |
| 7. Jenis kesalahan | prediksi salah kelas, salah lokasi, duplikat, latar, dan objek terlewat |
| 8. Eksperimen per kelompok gambar | AP pada gambar padat/sepi, kecil/besar, gelap/terang |
| 9. Domain shift | penurunan AP valid → test → test_own |
| 10. Galeri kegagalan | contoh visual gambar terburuk |
| 11. Kondisi sulit test_own | AP per kondisi foto (gelap, blur, oklusi, ...) |
| 12. Kerangka 5 analisis kegagalan | templat hipotesis → eksperimen → hasil |

**Cara menjalankan di Kaggle:**
1. *Settings* → *Accelerator*: **None** (CPU cukup), *Internet*: **On** (untuk `pip`).
2. *Add Input* → *Notebooks* → tambahkan output: **Notebook 0**, **Notebook 1**, **Notebook 2**, dan **Notebook 3 versi
   prediksi** (yang berisi `eval_test.json`).
3. **Save Version → Save & Run All.** Semua tabel disimpan sebagai CSV dan semua gambar di `figures/`.

**test_own:** bagian yang memakai test_own otomatis muncul setelah Notebook 0 dijalankan ulang dengan test set
buatanmu dan Notebook 1–3 dijalankan dengan `RUN_TRAINING = False`. Pastikan input di notebook ini memakai versi
terbaru keempat notebook tersebut. Kondisi foto dibaca dari awalan nama file (lihat `CONDITIONS` di konfigurasi).

In [ ]:
!pip install -q pycocotools

## 0. Konfigurasi

In [ ]:
import contextlib
import glob
import io
import json
import re
import shutil
import zipfile
from collections import defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

INPUT_ROOT = "/kaggle/input"                          # output Notebook 0–3 (lewat Add Input)
DATA_ZIP_GLOB = "/kaggle/input/**/indoor_seg_data.zip"
DATA_ROOT = Path("/tmp/data")
WORK = Path("/kaggle/working")

MODELS = {"yolo11m_seg": "YOLO11m-seg",                 # urutan ini = urutan warna, jangan diubah
          "mask_rcnn_r50": "Mask R-CNN R50-FPN v2",
          "mask2former_swin_t": "Mask2Former Swin-T"}
TAGS = ["025", "050", "100"]
FOCUS_TAG = "100"                 # fraksi yang dipakai untuk analisis kegagalan (bagian 5–11)
IOU_MATCH = 0.5                   # ambang IoU prediksi dianggap benar (bagian 7, 10)
RARE_MAX = 29                     # kelas dengan 1..RARE_MAX instance di train 100% = kelas langka
RUN_SUBSET_EXPERIMENTS = True     # bagian 8 dan 11: evaluasi ulang per kelompok gambar (beberapa menit)
N_SHOW = 4                        # banyaknya gambar terburuk yang ditampilkan per split

# test_own: kondisi dibaca dari awalan nama file sebelum "_" atau "-", mis. "gelap_03.jpg" -> "gelap".
# Awalan yang tidak ada di daftar ini masuk kelompok "lainnya".
CONDITIONS = ["normal", "gelap", "terang", "silau", "blur", "oklusi", "padat", "jauh", "dekat", "miring",
              "pantulan", "bayangan"]

COLORS = {"yolo11m_seg": "#2a78d6", "mask_rcnn_r50": "#eb6834", "mask2former_swin_t": "#1baf7a"}
MARKERS = {"yolo11m_seg": "o", "mask_rcnn_r50": "s", "mask2former_swin_t": "^"}
FRAC_SHADES = {"025": "#86b6ef", "050": "#2a78d6", "100": "#104281"}   # satu hue, terang -> gelap = data lebih banyak
OK_COLOR, ERR_COLOR = "#2a78d6", "#eb6834"                             # galeri: prediksi benar / salah
SHORT = {"yolo11m_seg": "YOLO", "mask_rcnn_r50": "MaskRCNN", "mask2former_swin_t": "M2F"}   # untuk tabel lebar

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning, message="invalid value encountered")   # korelasi nilai konstan

FIG_DIR = WORK / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#b5b4ad",
    "axes.grid": True, "grid.color": "#e8e7e2", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.labelcolor": "#3d3d3a", "xtick.color": "#5f5e5a", "ytick.color": "#5f5e5a", "text.color": "#2b2b29",
    "axes.titlesize": 11, "font.size": 10, "legend.frameon": False, "lines.linewidth": 2,
})
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)


def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=150)
    plt.show()


def show_table(df, name, floatfmt=3):
    """Cetak tabel dan simpan sebagai CSV di /kaggle/working."""
    df.to_csv(WORK / f"{name}.csv")
    print(df.round(floatfmt).to_string())

## Data dan hasil Notebook 1–3

In [ ]:
zips = sorted(glob.glob(DATA_ZIP_GLOB, recursive=True))
assert zips, "indoor_seg_data.zip tidak ditemukan. Tambahkan output Notebook 0 lewat Add Input."
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
with zipfile.ZipFile(zips[0]) as z:
    z.extractall(DATA_ROOT)
INFO = json.loads((DATA_ROOT / "dataset_info.json").read_text(encoding="utf-8"))


def load_coco(path):
    with contextlib.redirect_stdout(io.StringIO()):
        return COCO(str(path))


SPLITS = [s for s in ["valid", "test", "test_own"] if (DATA_ROOT / "annotations" / f"{s}.json").exists()]
GT = {s: load_coco(DATA_ROOT / "annotations" / f"{s}.json") for s in SPLITS}
GT["train"] = load_coco(DATA_ROOT / "annotations" / "train_100.json")
CAT_NAME = {c["id"]: c["name"] for c in GT["train"].dataset["categories"]}
TRAIN_COUNT = pd.Series(0, index=sorted(CAT_NAME))
for a in GT["train"].dataset["annotations"]:
    TRAIN_COUNT[a["category_id"]] += 1
N_TRAIN_IMG = {int(t): INFO["subsets"][t]["images"] for t in TAGS}
print("zip  :", zips[0])
print("split:", {s: len(GT[s].getImgIds()) for s in ["train"] + SPLITS}, "gambar")


def find_run(model, tag):
    """Folder run dengan file evaluasi terbanyak (kalau beberapa versi output ikut ditambahkan sebagai input)."""
    cands = [Path(p) for p in glob.glob(f"{INPUT_ROOT}/**/runs/{model}_{tag}", recursive=True)]
    cands = [c for c in cands if any(c.glob("eval_*.json"))]
    return max(cands, key=lambda c: (len(list(c.glob("eval_*.json"))), c.stat().st_mtime)) if cands else None


RUN_DIRS = {(m, t): find_run(m, t) for m in MODELS for t in TAGS}
print(pd.DataFrame([{"model": MODELS[m], "fraksi": t,
                     "split dengan hasil": ", ".join(sorted(p.stem[5:] for p in d.glob("eval_*.json"))) if d else "-",
                     "folder": str(d) if d else "TIDAK DITEMUKAN"} for (m, t), d in RUN_DIRS.items()]).to_string(index=False))
missing = [f"{MODELS[m]} {t}" for (m, t), d in RUN_DIRS.items() if d is None]
assert not missing, f"Output run tidak ditemukan: {missing}. Tambahkan output Notebook 1–3 lewat Add Input."
for s in SPLITS:
    lack = [f"{MODELS[m]} {t}" for (m, t), d in RUN_DIRS.items() if not (d / f"eval_{s}.json").exists()]
    if lack:
        print(f"PERINGATAN: belum ada hasil '{s}' untuk {lack}. Jalankan notebook model itu dengan RUN_TRAINING = False.")

In [ ]:
_PREDS, _EVAL = {}, {}


def preds(m, t, s):
    """Prediksi tersimpan (format hasil COCO). None kalau split ini belum diprediksi."""
    key = (m, t, s)
    if key not in _PREDS:
        p = RUN_DIRS[(m, t)] / f"preds_{s}.json"
        dets = json.loads(p.read_text()) if p.exists() else None
        if dets is not None:
            bad = {d["image_id"] for d in dets} - set(GT[s].getImgIds())
            assert not bad, (f"{p}: image_id {sorted(bad)[:5]} tidak ada di anotasi '{s}'. Prediksi dibuat dengan versi "
                             "Notebook 0 yang berbeda; jalankan ulang notebook model dengan RUN_TRAINING = False.")
        _PREDS[key] = dets
    return _PREDS[key]


def coco_eval(m, t, s, img_ids=None, iou_type="segm"):
    """Evaluasi COCO dari prediksi tersimpan, opsional hanya pada sebagian gambar.
    Sama dengan seg_common.py: untuk 'segm', bbox dibuang supaya area dihitung dari mask."""
    key = (m, t, s, iou_type)
    if img_ids is None and key in _EVAL:
        return _EVAL[key]
    dets = preds(m, t, s)
    if not dets:
        return None
    keep = None if img_ids is None else set(int(i) for i in img_ids)
    d = [{k: v for k, v in x.items() if not (iou_type == "segm" and k == "bbox")}
         for x in dets if keep is None or x["image_id"] in keep]
    if not d:
        return None
    with contextlib.redirect_stdout(io.StringIO()):
        ev = COCOeval(GT[s], GT[s].loadRes(d), iou_type)
        if keep is not None:
            ev.params.imgIds = sorted(keep)
        ev.evaluate()
        ev.accumulate()
        ev.summarize()
    out = {"stats": ev.stats.copy(), "precision": ev.eval["precision"].copy(), "catIds": list(ev.params.catIds)}
    if img_ids is None:
        _EVAL[key] = out
    return out


def ap(e):
    return float(e["stats"][0]) if e else np.nan

## 1. Ringkasan AP: 3 model × 3 fraksi data latih

Angka diambil dari `eval_<split>.json` yang ditulis Notebook 1–3 (fungsi `seg_common.py` yang sama). Mask AP =
COCO AP@[.50:.95] dari mask; box AP dihitung dari box prediksi. Fraksi 25/50/100% = 99/198/395 gambar train.

In [ ]:
rows, PER_CLASS = [], {}
for (m, t), d in RUN_DIRS.items():
    summ = json.loads((d / "train_summary.json").read_text()) if (d / "train_summary.json").exists() else {}
    for s in SPLITS:
        p = d / f"eval_{s}.json"
        if not p.exists():
            continue
        r = json.loads(p.read_text())
        PER_CLASS[(m, t, s)] = pd.Series(r["per_class_segm_AP"], dtype=float)
        rows.append({"model_key": m, "model": MODELS[m], "fraksi": int(t), "split": s,
                     **{k: v for k, v in r.items() if k.startswith(("segm_", "bbox_"))},
                     "ms_per_image": r.get("ms_per_image"), "train_hours": summ.get("train_hours"),
                     "epochs": summ.get("epochs_done")})
METRICS = pd.DataFrame(rows)
METRICS.to_csv(WORK / "ringkasan_metrik.csv", index=False)

for col, title in [("segm_AP", "Mask AP"), ("segm_AP50", "Mask AP50"), ("bbox_AP", "Box AP")]:
    print(f"\n{title} (baris = model, kolom = split / fraksi %)")
    t = METRICS.pivot_table(index="model", columns=["split", "fraksi"], values=col).reindex(MODELS.values())
    show_table(t[[c for c in SPLITS if c in t.columns.get_level_values(0)]], f"tabel_{col}")

In [ ]:
def end_labels(ax, points, x, gap):
    """Label nilai di ujung kanan garis; digeser vertikal seperlunya supaya tidak bertumpuk."""
    points = sorted(points)
    ys = [p[0] for p in points]
    for i in range(1, len(ys)):
        ys[i] = max(ys[i], ys[i - 1] + gap)
    for (y, txt), y_adj in zip(points, ys):
        ax.text(x, y_adj, txt, va="center", fontsize=9, color="#3d3d3a")


ymax = METRICS.segm_AP.max()
fig, axes = plt.subplots(1, len(SPLITS), figsize=(4.3 * len(SPLITS), 3.9), sharey=True, squeeze=False)
for ax, s in zip(axes[0], SPLITS):
    ends = []
    for m in MODELS:
        d = METRICS[(METRICS.model_key == m) & (METRICS.split == s)].sort_values("fraksi")
        if d.empty:
            continue
        ax.plot(d.fraksi, d.segm_AP, color=COLORS[m], marker=MARKERS[m], ms=7, label=MODELS[m])
        ends.append((d.segm_AP.iloc[-1], f"{d.segm_AP.iloc[-1]:.3f}"))
    end_labels(ax, ends, 104, 0.045 * ymax)
    ax.set_title(s)
    ax.set_xticks([25, 50, 100], [f"{f}%\n({N_TRAIN_IMG[f]} gbr)" for f in (25, 50, 100)])
    ax.set_xlim(15, 118)
    ax.set_xlabel("data latih")
axes[0][0].set_ylabel("mask AP")
axes[0][0].set_ylim(bottom=0)
fig.legend(*axes[0][0].get_legend_handles_labels(), loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.07))
fig.suptitle("Mask AP terhadap jumlah data latih", y=1.13, fontsize=12)
save(fig, "01_ap_vs_fraksi")

rows = []
for m in MODELS:
    for s in SPLITS:
        d = METRICS[(METRICS.model_key == m) & (METRICS.split == s)].set_index("fraksi").segm_AP
        if {25, 50, 100} <= set(d.index):
            rows.append({"model": MODELS[m], "split": s, "AP 25%": d[25], "AP 50%": d[50], "AP 100%": d[100],
                         "naik 25→50": d[50] - d[25], "naik 50→100": d[100] - d[50],
                         "naik 25→100 (relatif %)": 100 * (d[100] - d[25]) / max(d[25], 1e-9)})
print("Kenaikan mask AP karena tambahan data latih:")
show_table(pd.DataFrame(rows).set_index(["model", "split"]), "kenaikan_ap_per_fraksi")

In [ ]:
rows = []
for m in MODELS:
    r = {"model": MODELS[m]}
    for t in TAGS:
        x = METRICS[(METRICS.model_key == m) & (METRICS.fraksi == int(t))]
        r[f"jam latih {int(t)}%"] = x.train_hours.iloc[0] if len(x) else np.nan
        r[f"epoch {int(t)}%"] = x.epochs.iloc[0] if len(x) else np.nan
    x = METRICS[(METRICS.model_key == m) & (METRICS.fraksi == int(FOCUS_TAG))].set_index("split")
    r["ms/gambar (test)"] = x.ms_per_image.get("test", np.nan)
    for s in SPLITS:
        r[f"mask AP {s}"] = x.segm_AP.get(s, np.nan)
    rows.append(r)
print(f"Biaya vs hasil (fraksi {int(FOCUS_TAG)}% untuk AP dan kecepatan; T4, satu gambar per panggilan, termasuk "
      "upsample mask ke ukuran asli):")
show_table(pd.DataFrame(rows).set_index("model"), "biaya_vs_hasil", 3)

## 2. Kurva belajar

YOLO mencatat mAP50-95 mask dari validator Ultralytics setiap epoch (bukan pycocotools, jadi angkanya sedikit beda
dari tabel di atas). Mask R-CNN dan Mask2Former mencatat mask AP valid pycocotools setiap 2 epoch. Titik = epoch
dengan skor valid tertinggi (bobot yang dipakai untuk evaluasi).

In [ ]:
def learning_curve(d):
    df = pd.read_csv(d / "results.csv")
    df.columns = [c.strip() for c in df.columns]
    if "metrics/mAP50-95(M)" in df:
        return df["epoch"], df["metrics/mAP50-95(M)"], "mask mAP50-95 valid (Ultralytics)"
    df = df.dropna(subset=["val_segm_AP"])
    return df["epoch"], df["val_segm_AP"], "mask AP valid (pycocotools)"


fig, axes = plt.subplots(1, len(MODELS), figsize=(4.4 * len(MODELS), 3.6), squeeze=False)
rows = []
for ax, m in zip(axes[0], MODELS):
    for t in TAGS:
        x, y, lab = learning_curve(RUN_DIRS[(m, t)])
        ax.plot(x, y, color=FRAC_SHADES[t], label=f"{int(t)}%")
        i = int(np.nanargmax(y.values))
        ax.plot(x.iloc[i], y.iloc[i], "o", color=FRAC_SHADES[t], ms=8, mec="white", mew=1.5)
        n = max(2, len(y) // 10)                          # 10% evaluasi terakhir vs 10% sebelumnya
        rows.append({"model": MODELS[m], "fraksi": int(t), "epoch terbaik": int(x.iloc[i]), "epoch total": int(x.iloc[-1]),
                     "skor terbaik": y.iloc[i], "skor akhir": y.iloc[-1],
                     "tren 10% terakhir": y.iloc[-n:].mean() - y.iloc[-2 * n:-n].mean()})
    ax.set_title(MODELS[m])
    ax.set_xlabel("epoch")
    ax.set_ylabel(lab)
axes[0][0].legend(title="data latih")
save(fig, "02_kurva_belajar")
print("'tren 10% terakhir' > ~0,005 berarti skor valid masih naik di akhir training (epoch bisa ditambah).")
show_table(pd.DataFrame(rows).set_index(["model", "fraksi"]), "kurva_belajar_ringkas", 4)

## 3. Statistik split: seberapa beda datanya?

Dihitung langsung dari gambar dan anotasi. Kecerahan dan kontras = rata-rata dan simpangan baku piksel abu-abu;
ketajaman = varians Laplacian setelah gambar diperkecil ke sisi panjang 640 (supaya resolusi asli tidak memengaruhi).
JS divergence mengukur beda distribusi kelas terhadap train (0 = identik, 1 = tidak beririsan).

In [ ]:
def image_stats(split):
    gt, rows = GT[split], []
    folder = DATA_ROOT / "images" / ("train" if split == "train" else split)
    for im in gt.dataset["images"]:
        with Image.open(folder / im["file_name"]) as f:
            f.draft("L", (640, 640))
            g = f.convert("L")
            g.thumbnail((640, 640))
        a = np.asarray(g, dtype=np.float32)
        rows.append({"split": split, "image_id": im["id"], "file_name": im["file_name"], "width": im["width"],
                     "height": im["height"], "mp": im["width"] * im["height"] / 1e6, "brightness": a.mean(),
                     "contrast": a.std(), "sharpness": cv2.Laplacian(a, cv2.CV_32F).var(),
                     "n_inst": len(gt.getAnnIds(imgIds=im["id"]))})
    return rows


IMG = pd.DataFrame([r for s in ["train"] + SPLITS for r in image_stats(s)])
IMG.to_csv(WORK / "statistik_gambar.csv", index=False)


def class_dist(gt):
    c = np.zeros(max(CAT_NAME) + 1)
    for a in gt.dataset["annotations"]:
        c[a["category_id"]] += 1
    return c / max(c.sum(), 1)


def js_div(p, q):
    mid = (p + q) / 2
    kl = lambda a, b: float(np.sum(a[a > 0] * np.log2(a[a > 0] / b[a > 0])))
    return (kl(p, mid) + kl(q, mid)) / 2


rare = set(TRAIN_COUNT[(TRAIN_COUNT > 0) & (TRAIN_COUNT <= RARE_MAX)].index)
unseen = set(TRAIN_COUNT[TRAIN_COUNT == 0].index)
p_train = class_dist(GT["train"])
rows = []
for s in ["train"] + SPLITS:
    g, anns = IMG[IMG.split == s], GT[s].dataset["annotations"]
    cats = np.array([a["category_id"] for a in anns])
    rel_area = np.array([a["area"] / (GT[s].imgs[a["image_id"]]["width"] * GT[s].imgs[a["image_id"]]["height"])
                         for a in anns])
    rows.append({"split": s, "gambar": len(g), "instance": len(anns), "inst/gambar": g.n_inst.mean(),
                 "maks inst/gambar": g.n_inst.max(), "MP median": g.mp.median(), "MP maks": g.mp.max(),
                 "% gambar >2 MP": 100 * (g.mp > 2).mean(), "kecerahan": g.brightness.mean(),
                 "kontras": g.contrast.mean(), "ketajaman median": g.sharpness.median(),
                 "objek <1% gambar (%)": 100 * (rel_area < 0.01).mean(),
                 "JS div kelas vs train": js_div(class_dist(GT[s]), p_train),
                 "% inst kelas langka": 100 * np.isin(cats, list(rare)).mean(),
                 "% inst kelas tak ada di train": 100 * np.isin(cats, list(unseen)).mean()})
SPLIT_STATS = pd.DataFrame(rows).set_index("split")
show_table(SPLIT_STATS.T, "statistik_split", 3)

order = ["train"] + SPLITS
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
for ax, (col, lab, log) in zip(axes, [("mp", "megapiksel", True), ("brightness", "kecerahan (0–255)", False),
                                      ("sharpness", "ketajaman (var. Laplacian)", True),
                                      ("n_inst", "instance per gambar", False)]):
    data = [IMG.loc[IMG.split == s, col].values for s in order]
    ax.boxplot(data, widths=0.5, patch_artist=True, showfliers=True,
               boxprops=dict(facecolor="#d9d8d2", edgecolor="#5f5e5a"), medianprops=dict(color="#2b2b29", lw=2),
               whiskerprops=dict(color="#5f5e5a"), capprops=dict(color="#5f5e5a"),
               flierprops=dict(marker="o", ms=3, mfc="#8a8983", mec="none", alpha=0.6))
    ax.set_xticks(range(1, len(order) + 1), order)
    ax.set_title(lab)
    if log:
        ax.set_yscale("log")
fig.suptitle("Distribusi per split", y=1.03, fontsize=12)
save(fig, "03_statistik_split")

## 4. Frekuensi kelas di train vs AP

Kelas dikelompokkan menurut jumlah instance di train 100%. Rata-rata AP per kelompok hanya memakai kelas yang ada
di split tersebut (sama seperti mAP COCO). Ini eksperimen pembuktian untuk kegagalan "kelas langka".

In [ ]:
def group_of(cid):
    n = TRAIN_COUNT[cid]
    return "0 (tak ada di train)" if n == 0 else (f"1–{RARE_MAX} (langka)" if n <= RARE_MAX else f">{RARE_MAX}")


GROUP_ORDER = ["0 (tak ada di train)", f"1–{RARE_MAX} (langka)", f">{RARE_MAX}"]
name2id = {v: k for k, v in CAT_NAME.items()}
for s in SPLITS:
    present = {a["category_id"] for a in GT[s].dataset["annotations"]}
    rows = []
    for grp in GROUP_ORDER:
        ids = [c for c in present if group_of(c) == grp]
        r = {"kelompok": grp, "kelas": len(ids), "instance": sum(len(GT[s].getAnnIds(catIds=[c])) for c in ids)}
        for m in MODELS:
            for t in TAGS:
                pc = PER_CLASS.get((m, t, s))
                r[(SHORT[m], int(t))] = pc[[CAT_NAME[c] for c in ids]].mean() if pc is not None and ids else np.nan
        rows.append(r)
    t = pd.DataFrame(rows).set_index(["kelompok", "kelas", "instance"])
    t.columns = pd.MultiIndex.from_tuples(t.columns, names=["model", "fraksi %"])
    print(f"\n[{s}] mask AP rata-rata per kelompok kelas")
    show_table(t, f"ap_per_kelompok_kelas_{s}", 3)

split_scatter = [s for s in SPLITS if s != "valid"] or SPLITS
fig, axes = plt.subplots(len(split_scatter), len(MODELS), figsize=(4.2 * len(MODELS), 3.6 * len(split_scatter)),
                         sharex=True, sharey=True, squeeze=False, constrained_layout=True)
ticks = [t for t in [0, 1, 3, 10, 30, 100, 300, 1000, 3000] if t <= TRAIN_COUNT.max() * 1.5]
for row, s in zip(axes, split_scatter):
    present = sorted({a["category_id"] for a in GT[s].dataset["annotations"]})
    for ax, m in zip(row, MODELS):
        pc = PER_CLASS.get((m, FOCUS_TAG, s))
        if pc is None:
            ax.set_visible(False)
            continue
        x = TRAIN_COUNT[present].values
        y = pc[[CAT_NAME[c] for c in present]].values
        ok = ~np.isnan(y)
        rho = pd.Series(x[ok]).rank().corr(pd.Series(y[ok]).rank())
        ax.scatter(x[ok], y[ok], s=28, color=COLORS[m], edgecolor="white", linewidth=0.8, marker=MARKERS[m])
        ax.set_xscale("symlog", linthresh=1)
        ax.set_xticks(ticks, [str(t) for t in ticks])
        ax.set_title(f"{MODELS[m]} — {s}\nSpearman ρ = {rho:.2f}")
    row[0].set_ylabel(f"mask AP per kelas ({int(FOCUS_TAG)}%)")
for ax in axes[-1]:
    ax.set_xlabel("instance kelas di train 100%")
save(fig, "04_ap_vs_frekuensi_kelas")

## 5. AP terhadap ambang IoU (kualitas tepi mask)

AP dihitung terpisah di setiap ambang IoU 0,50…0,95. Kurva yang turun curam berarti mask sering "kira-kira benar"
tetapi tepinya kurang presisi. Relevan untuk membandingkan resolusi mask: Mask R-CNN 28×28 per RoI,
YOLO prototipe 1/4 input, Mask2Former 1/4 input per query.

In [ ]:
IOUS = np.linspace(0.5, 0.95, 10)


def ap_per_iou(e):
    p = e["precision"][:, :, :, 0, 2]                     # IoU x recall x kelas, area=all, maxDets=100
    return np.array([p[i][p[i] > -1].mean() if (p[i] > -1).any() else np.nan for i in range(len(IOUS))])


fig, axes = plt.subplots(1, len(SPLITS), figsize=(4.3 * len(SPLITS), 3.6), sharey=True, squeeze=False)
rows = []
for ax, s in zip(axes[0], SPLITS):
    for m in MODELS:
        e = coco_eval(m, FOCUS_TAG, s)
        if e is None:
            continue
        a = ap_per_iou(e)
        ax.plot(IOUS, a, color=COLORS[m], marker=MARKERS[m], ms=6, label=MODELS[m])
        ref = METRICS[(METRICS.model_key == m) & (METRICS.fraksi == int(FOCUS_TAG)) & (METRICS.split == s)].segm_AP
        rows.append({"model": MODELS[m], "split": s, "AP@.50": a[0], "AP@.75": a[5], "AP@.90": a[8],
                     "AP@.75 / AP@.50": a[5] / a[0] if a[0] else np.nan,
                     "cek: AP = tabel bag. 1": bool(len(ref)) and bool(np.isclose(ap(e), ref.iloc[0], atol=1e-4))})
    ax.set_title(s)
    ax.set_xlabel("ambang IoU")
axes[0][0].set_ylabel(f"mask AP ({int(FOCUS_TAG)}%)")
fig.legend(*axes[0][0].get_legend_handles_labels(), loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.07))
save(fig, "05_ap_vs_iou")
print("Rasio AP@.75 / AP@.50 lebih kecil = tepi mask kurang presisi. Kolom 'cek' harus True semua "
      "(evaluasi ulang di sini identik dengan Notebook 1–3).")
show_table(pd.DataFrame(rows).set_index(["model", "split"]), "ap_per_iou", 3)

## 6. Ukuran objek

APs/APm/APl memakai batas area COCO absolut (32² dan 96² piksel). Pada gambar beresolusi tinggi hampir semua objek
masuk "besar", jadi tabel kedua memakai ukuran relatif (luas mask / luas gambar) untuk membandingkan split.

In [ ]:
cols = {"segm_APs": "AP kecil", "segm_APm": "AP sedang", "segm_APl": "AP besar"}
t = METRICS[METRICS.fraksi == int(FOCUS_TAG)].set_index(["model", "split"])[list(cols)].rename(columns=cols)
print(f"Mask AP per ukuran objek (fraksi {int(FOCUS_TAG)}%; -1 = tidak ada objek GT seukuran itu):")
show_table(t, "ap_per_ukuran", 3)

rows = []
for s in ["train"] + SPLITS:
    anns = GT[s].dataset["annotations"]
    absa = np.array([a["area"] for a in anns])
    rel = np.array([a["area"] / (GT[s].imgs[a["image_id"]]["width"] * GT[s].imgs[a["image_id"]]["height"]) for a in anns])
    rows.append({"split": s, "kecil <32² px (%)": 100 * (absa < 32 ** 2).mean(),
                 "sedang (%)": 100 * ((absa >= 32 ** 2) & (absa < 96 ** 2)).mean(), "besar ≥96² px (%)": 100 * (absa >= 96 ** 2).mean(),
                 "<0,1% gambar (%)": 100 * (rel < 0.001).mean(), "0,1–1% gambar (%)": 100 * ((rel >= 0.001) & (rel < 0.01)).mean(),
                 "≥1% gambar (%)": 100 * (rel >= 0.01).mean()})
print("\nSebaran ukuran objek GT per split:")
show_table(pd.DataFrame(rows).set_index("split"), "sebaran_ukuran_objek", 1)

## 7. Jenis kesalahan (pada fraksi fokus)

Setiap prediksi dicocokkan secara greedy (skor tertinggi dulu) ke mask GT:
**benar** (IoU ≥ `IOU_MATCH`, kelas sama, GT belum terpakai), **duplikat** (GT sudah terpakai),
**salah kelas** (IoU ≥ `IOU_MATCH` dengan GT kelas lain), **lokalisasi** (kelas sama, 0,1 ≤ IoU < `IOU_MATCH`),
**latar** (tidak mengenai objek apa pun). GT yang tidak mendapat prediksi benar = **terlewat**.

Ambang skor tiap model dipilih di **valid** (F1 tertinggi) lalu dipakai apa adanya di split lain, karena skala skor
ketiga model berbeda (Mask2Former: prob kelas × skor mask).

In [ ]:
def match_preds(m, t, s, score_min=0.05):
    gt, dets = GT[s], preds(m, t, s)
    by_img = defaultdict(list)
    for i, d in enumerate(dets or []):
        if d["score"] >= score_min:
            by_img[d["image_id"]].append(i)
    rows = []
    for img_id in gt.getImgIds():
        idx = sorted(by_img.get(img_id, []), key=lambda i: -dets[i]["score"])
        if not idx:
            continue
        anns = gt.loadAnns(gt.getAnnIds(imgIds=img_id))
        g_cat = np.array([a["category_id"] for a in anns])
        if anns:
            iou = np.asarray(mask_utils.iou([dets[i]["segmentation"] for i in idx], [gt.annToRLE(a) for a in anns],
                                            [0] * len(anns))).reshape(len(idx), len(anns))
        else:
            iou = np.zeros((len(idx), 0))
        used = np.zeros(len(anns), bool)
        for k, i in enumerate(idx):
            d, same = dets[i], g_cat == dets[i]["category_id"]
            cand = np.where(same & ~used, iou[k], -1.0)
            j = int(cand.argmax()) if len(cand) else -1
            gt_id = -1
            if j >= 0 and cand[j] >= IOU_MATCH:
                kind, used[j], gt_id = "benar", True, anns[j]["id"]
            elif (iou[k][same] >= IOU_MATCH).any():
                kind = "duplikat"
            elif (iou[k][~same] >= IOU_MATCH).any():
                kind = "salah kelas"
            elif (iou[k][same] >= 0.1).any():
                kind = "lokalisasi"
            else:
                kind = "latar"
            rows.append((img_id, i, d["category_id"], d["score"], kind, gt_id))
    return pd.DataFrame(rows, columns=["image_id", "idx", "category_id", "score", "kind", "gt_id"])


def f1_at(df, n_gt, thr):
    k = df[df.score >= thr]
    tp = (k.kind == "benar").sum()
    return 2 * tp / max(len(k) + n_gt, 1)


N_GT = {s: len(GT[s].getAnnIds()) for s in SPLITS}
MATCH, THRESH = {}, {}
for m in MODELS:
    for s in SPLITS:
        if preds(m, FOCUS_TAG, s):
            MATCH[(m, s)] = match_preds(m, FOCUS_TAG, s)
    grid = np.round(np.arange(0.05, 0.96, 0.05), 2)
    f1 = [f1_at(MATCH[(m, "valid")], N_GT["valid"], g) for g in grid]
    THRESH[m] = float(grid[int(np.argmax(f1))])
print("Ambang skor (F1 terbaik di valid):", {MODELS[m]: v for m, v in THRESH.items()})

KINDS = ["benar", "salah kelas", "lokalisasi", "duplikat", "latar"]
rows = []
for (m, s), df in MATCH.items():
    k = df[df.score >= THRESH[m]]
    c = k.kind.value_counts().reindex(KINDS, fill_value=0)
    tp, n_pred = c["benar"], len(k)
    rows.append({"model": MODELS[m], "split": s, "prediksi": n_pred, **{f"{x}": c[x] for x in KINDS},
                 "terlewat": N_GT[s] - tp, "presisi": tp / max(n_pred, 1), "recall": tp / max(N_GT[s], 1),
                 "F1": 2 * tp / max(n_pred + N_GT[s], 1),
                 **{f"% salah: {x}": 100 * c[x] / max(n_pred - tp, 1) for x in KINDS[1:]}})
ERRORS = pd.DataFrame(rows).set_index(["model", "split"])
show_table(ERRORS, "jenis_kesalahan", 3)

rows = []                                              # tabel per gambar, dipakai bagian 8–11
for (m, s), df in MATCH.items():
    k = df[df.score >= THRESH[m]]
    g = k.groupby("image_id").kind.agg(tp=lambda x: (x == "benar").sum(), n_pred="size")
    base = IMG[IMG.split == s].set_index("image_id")
    g = g.reindex(base.index, fill_value=0)
    g["fn"] = base.n_inst - g.tp
    g["fp"] = g.n_pred - g.tp
    g["f1"] = 2 * g.tp / (2 * g.tp + g.fp + g.fn).replace(0, np.nan)
    g["model_key"], g["split"] = m, s
    rows.append(g.reset_index())
PER_IMAGE = pd.concat(rows).merge(IMG, on=["split", "image_id"])
PER_IMAGE.to_csv(WORK / "per_gambar.csv", index=False)

print("\nKorelasi Spearman F1 per gambar dengan sifat gambar (negatif = makin besar nilainya, makin buruk):")
rows = []
for (m, s), g in PER_IMAGE.groupby(["model_key", "split"]):
    g = g.dropna(subset=["f1"])
    rows.append({"model": MODELS[m], "split": s, **{c: g.f1.rank().corr(g[c].rank())
                 for c in ["n_inst", "mp", "brightness", "contrast", "sharpness"]}})
show_table(pd.DataFrame(rows).set_index(["model", "split"]), "korelasi_f1_sifat_gambar", 2)

## 8. Eksperimen per kelompok gambar

Gambar dikelompokkan menurut satu sifat, lalu mask AP dihitung ulang hanya pada kelompok itu (pycocotools,
`params.imgIds`). Jumlah gambar per kelompok ikut ditampilkan; kelompok kecil = angka kurang stabil. Campuran kelas
antar kelompok bisa berbeda, jadi bandingkan **antar model di kelompok yang sama** dan **pola naik/turun**.

In [ ]:
def bucket_table(split, col, bins=None, labels=None, q=None):
    sub = IMG[IMG.split == split].copy()
    sub["kelompok"] = pd.qcut(sub[col], q, labels=labels, duplicates="drop") if q else pd.cut(sub[col], bins, labels=labels)
    rows = []
    for lab, g in sub.groupby("kelompok", observed=True):
        r = {"kelompok": lab, "gambar": len(g), "instance": int(g.n_inst.sum())}
        for m in MODELS:
            r[MODELS[m]] = ap(coco_eval(m, FOCUS_TAG, split, img_ids=g.image_id.tolist()))
        rows.append(r)
    return pd.DataFrame(rows).set_index("kelompok")


EXPERIMENTS = {
    "kepadatan (instance per gambar)": dict(col="n_inst", bins=[-1, 5, 15, 30, np.inf], labels=["0–5", "6–15", "16–30", ">30"]),
    "resolusi gambar": dict(col="mp", bins=[0, 0.5, 2, np.inf], labels=["≤0,5 MP", "0,5–2 MP", ">2 MP"]),
    "kecerahan (tertil per split)": dict(col="brightness", q=3, labels=["gelap", "sedang", "terang"]),
    "ketajaman (tertil per split)": dict(col="sharpness", q=3, labels=["buram", "sedang", "tajam"]),
}
if RUN_SUBSET_EXPERIMENTS:
    for s in [x for x in SPLITS if x != "valid"]:
        for name, kw in EXPERIMENTS.items():
            print(f"\n[{s}] mask AP per kelompok — {name} (fraksi {int(FOCUS_TAG)}%)")
            show_table(bucket_table(s, **kw), f"eksperimen_{s}_{kw['col']}", 3)
else:
    print("RUN_SUBSET_EXPERIMENTS = False, bagian ini dilewati.")

## 9. Domain shift: valid → test → test_own

Valid berasal dari distribusi yang sama dengan train. Penurunan AP ke test (split Roboflow dari sumber lain) dan ke
test_own (foto HP sendiri) mengukur seberapa jauh model bertahan di luar distribusi latih.

In [ ]:
rows = []
for m in MODELS:
    x = METRICS[(METRICS.model_key == m) & (METRICS.fraksi == int(FOCUS_TAG))].set_index("split").segm_AP
    r = {"model": MODELS[m], **{f"AP {s}": x.get(s, np.nan) for s in SPLITS}}
    for s in SPLITS[1:]:
        r[f"turun valid→{s}"] = x.get("valid", np.nan) - x.get(s, np.nan)
        r[f"sisa {s}/valid (%)"] = 100 * x.get(s, np.nan) / x.get("valid", np.nan)
    rows.append(r)
print(f"Mask AP fraksi {int(FOCUS_TAG)}% dan penurunannya:")
show_table(pd.DataFrame(rows).set_index("model"), "domain_shift", 3)

for s in SPLITS[1:]:
    both = [c for c in CAT_NAME if len(GT["valid"].getAnnIds(catIds=[c])) >= 5 and len(GT[s].getAnnIds(catIds=[c])) >= 5]
    rows = []
    for c in both:
        r = {"kelas": CAT_NAME[c], "inst train": TRAIN_COUNT[c], "inst valid": len(GT["valid"].getAnnIds(catIds=[c])),
             f"inst {s}": len(GT[s].getAnnIds(catIds=[c]))}
        for m in MODELS:
            r[MODELS[m]] = PER_CLASS[(m, FOCUS_TAG, "valid")][CAT_NAME[c]] - PER_CLASS[(m, FOCUS_TAG, s)][CAT_NAME[c]]
        rows.append(r)
    if not rows:
        continue
    t = pd.DataFrame(rows).set_index("kelas")
    t["rata-rata turun"] = t[list(MODELS.values())].mean(axis=1)
    print(f"\nKelas dengan penurunan AP valid→{s} terbesar (≥5 instance di kedua split; angka = AP valid − AP {s}):")
    show_table(t.sort_values("rata-rata turun", ascending=False).head(12), f"kelas_turun_{s}", 3)

## 10. Galeri kegagalan

Gambar dengan F1 rata-rata ketiga model terendah (minimal 3 objek GT; kalau F1 sama, yang objeknya lebih banyak
didahulukan). Panel kiri = GT dengan nama kelas, dan daftar objek GT (kelas, jumlah contohnya di train, luas) dicetak
di atas gambar. Panel lain = prediksi di atas ambang skor tiap model. **Biru** = prediksi benar, **oranye** = prediksi salah (label = kelas tebakan),
**garis putus-putus** = objek GT yang terlewat. Untuk menampilkan gambar lain: `show_image("test", image_id)`.

In [ ]:
def _resize_mask(mask, size):
    return cv2.resize(mask.astype(np.uint8), size, interpolation=cv2.INTER_NEAREST).astype(bool)


def _draw(ax, img, items, missed=()):
    out = img.astype(np.float32) / 255
    for mask, color, _ in items:
        rgb = np.array([int(color[i:i + 2], 16) for i in (1, 3, 5)]) / 255
        out[mask] = 0.55 * out[mask] + 0.45 * rgb
    ax.imshow(out)
    for mask, color, label in items:
        ax.contour(mask, levels=[0.5], colors=[color], linewidths=1)
        if label:
            ys, xs = np.nonzero(mask)
            ax.text(xs.mean(), ys.mean(), label, fontsize=7, color="#2b2b29", ha="center", va="center",
                    bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8))
    for mask in missed:
        ax.contour(mask, levels=[0.5], colors=["white"], linewidths=2.2)
        ax.contour(mask, levels=[0.5], colors=["#2b2b29"], linewidths=1, linestyles="dashed")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)


def show_image(split, image_id, tag=FOCUS_TAG, max_side=800, name=None):
    gt, im = GT[split], GT[split].imgs[image_id]
    img = np.asarray(Image.open(DATA_ROOT / "images" / split / im["file_name"]).convert("RGB"))
    sc = min(1.0, max_side / max(img.shape[:2]))
    size = (max(1, round(img.shape[1] * sc)), max(1, round(img.shape[0] * sc)))
    small = cv2.resize(img, size, interpolation=cv2.INTER_AREA) if sc < 1 else img
    anns = gt.loadAnns(gt.getAnnIds(imgIds=image_id))
    gmask = {a["id"]: _resize_mask(gt.annToMask(a), size) for a in anns}
    w = 4.0 * small.shape[1] / small.shape[0]
    fig, axes = plt.subplots(1, 1 + len(MODELS), figsize=(min(w, 6) * (1 + len(MODELS)), 4.4), squeeze=False)
    _draw(axes[0][0], small, [(gmask[a["id"]], OK_COLOR, CAT_NAME[a["category_id"]]) for a in anns])
    axes[0][0].set_title(f"GT: {len(anns)} objek\n{split} id={image_id}", fontsize=9)
    area = im["width"] * im["height"]
    gt_list = pd.DataFrame([{"kelas GT": CAT_NAME[a["category_id"]], "instance kelas di train": TRAIN_COUNT[a["category_id"]],
                             "% luas gambar": 100 * a["area"] / area} for a in anns])
    print(f"{split} id={image_id} ({im['file_name']}, {im['width']}x{im['height']}):")
    print(gt_list.sort_values("% luas gambar", ascending=False).round(1).to_string(index=False))
    for ax, m in zip(axes[0][1:], MODELS):
        if (m, split) not in MATCH:
            ax.set_visible(False)
            continue
        df = MATCH[(m, split)]
        k = df[(df.image_id == image_id) & (df.score >= THRESH[m])]
        dets = preds(m, tag, split)
        items = [(_resize_mask(mask_utils.decode(dets[r.idx]["segmentation"]), size),
                  OK_COLOR if r.kind == "benar" else ERR_COLOR,
                  None if r.kind == "benar" else CAT_NAME[r.category_id]) for r in k.itertuples()]
        hit = set(k.gt_id[k.kind == "benar"])
        missed = [gmask[a["id"]] for a in anns if a["id"] not in hit]
        _draw(ax, small, items, missed)
        tp = len(hit)
        ax.set_title(f"{MODELS[m]} (skor ≥ {THRESH[m]:.2f})\nbenar {tp} · salah {len(k) - tp} · terlewat {len(missed)}",
                     fontsize=9)
    if name:
        save(fig, name)
    else:
        plt.show()


for s in [x for x in SPLITS if x != "valid"]:
    agg = (PER_IMAGE[PER_IMAGE.split == s].groupby("image_id")
           .agg(f1=("f1", "mean"), n_inst=("n_inst", "first"), file_name=("file_name", "first")))
    worst = agg[agg.n_inst >= 3].sort_values(["f1", "n_inst"], ascending=[True, False]).head(N_SHOW)
    print(f"\n[{s}] {N_SHOW} gambar terburuk (F1 rata-rata ketiga model):")
    print(worst.round(3).to_string())
    for i, img_id in enumerate(worst.index):
        show_image(s, int(img_id), name=f"10_galeri_{s}_{i + 1}")

## 11. Kondisi sulit pada test_own

Kondisi dibaca dari awalan nama file foto (sebelum `_` atau `-`), mis. `gelap_03.jpg`, `blur-12.jpg`,
`normal_01.jpg`. Ganti `CONDITIONS` di konfigurasi kalau memakai nama lain. Kelompok `normal` menjadi pembanding.

In [ ]:
def condition(file_name):
    m = re.match(r"([A-Za-z]+)[_\-]", file_name)
    c = m.group(1).lower() if m else ""
    return c if c in CONDITIONS else "lainnya"


if "test_own" not in SPLITS:
    print("test_own belum ada. Bagian ini terisi setelah Notebook 0 dijalankan dengan test set buatanmu.")
else:
    own = IMG[IMG.split == "test_own"].copy()
    own["kondisi"] = own.file_name.map(condition)
    if (own.kondisi == "lainnya").all():
        print("PERINGATAN: tidak ada nama file yang diawali kondisi dari CONDITIONS. Contoh nama file:",
              own.file_name.head(3).tolist())
    rows = []
    for cond, g in own.groupby("kondisi"):
        r = {"kondisi": cond, "foto": len(g), "instance": int(g.n_inst.sum()), "kecerahan": g.brightness.mean(),
             "ketajaman median": g.sharpness.median()}
        for m in MODELS:
            if RUN_SUBSET_EXPERIMENTS:
                r[f"AP {MODELS[m]}"] = ap(coco_eval(m, FOCUS_TAG, "test_own", img_ids=g.image_id.tolist()))
            pi = PER_IMAGE[(PER_IMAGE.model_key == m) & (PER_IMAGE.split == "test_own") & PER_IMAGE.image_id.isin(g.image_id)]
            r[f"F1 {MODELS[m]}"] = pi.f1.mean()
        rows.append(r)
    COND = pd.DataFrame(rows).set_index("kondisi")
    show_table(COND, "test_own_per_kondisi", 3)

    metric = "AP" if RUN_SUBSET_EXPERIMENTS else "F1"
    conds = COND.index.tolist()
    xs = np.arange(len(conds))
    wbar = 0.8 / len(MODELS)
    fig, ax = plt.subplots(figsize=(max(6, 1.3 * len(conds) + 2), 3.8))
    for k, m in enumerate(MODELS):
        vals = COND[f"{metric} {MODELS[m]}"].values
        ax.bar(xs + (k - (len(MODELS) - 1) / 2) * wbar, vals, width=wbar - 0.03, color=COLORS[m], label=MODELS[m])
    ax.set_xticks(xs, [f"{c}\n({n} foto)" for c, n in zip(conds, COND.foto)])
    ax.set_ylabel(f"mask {metric} ({int(FOCUS_TAG)}%)")
    ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.15))
    ax.grid(axis="x", visible=False)
    save(fig, "11_test_own_per_kondisi")

## 12. Kerangka 5 analisis kegagalan

Tugas meminta 5 kegagalan, masing-masing dengan eksperimen pembuktian. Kandidat di bawah sudah punya bukti di
notebook ini; pilih 5, isi angkanya dari tabel/gambar yang disebut, lalu tulis kesimpulan. Contoh visual ambil dari
bagian 10 atau `show_image(...)`.

| # | Kegagalan (hipotesis) | Eksperimen pembuktian di notebook ini | Yang dilihat |
|---|---|---|---|
| A | Kelas yang tidak ada / langka di train hampir tidak pernah tersegmentasi benar | Bag. 4: AP per kelompok frekuensi + korelasi Spearman | AP kelompok 0 dan langka vs >RARE_MAX; ρ positif |
| B | Performa turun di data dari sumber lain (domain shift) | Bag. 3 (statistik split), bag. 9 (penurunan AP), bag. 8 resolusi | beda MP/kecerahan/JS div; AP valid vs test vs test_own |
| C | Tambahan data tidak menolong YOLO di split test (kurva datar) | Bag. 1 (kenaikan per fraksi), bag. 2 (kurva belajar), bag. 4 per kelompok | YOLO datar di test tetapi naik di valid; model lain naik |
| D | Tepi mask kurang presisi (resolusi mask) | Bag. 5: AP vs ambang IoU | rasio AP@.75/AP@.50 per arsitektur |
| E | Adegan padat dan objek kecil | Bag. 6 (AP per ukuran), bag. 8 kepadatan, korelasi `n_inst` di bag. 7 | AP turun saat instance per gambar naik |
| F | Salah kelas antar kelas mirip (mis. table vs dining table, paper vs paper-merged) | Bag. 7: porsi "salah kelas" + galeri | % salah kelas; contoh label oranye |
| G | Kondisi sulit foto HP (gelap, blur, oklusi, ...) | Bag. 11 (AP per kondisi), bag. 8 kecerahan/ketajaman | AP kondisi vs `normal` |

Format per kegagalan untuk laporan:
1. **Gejala** — contoh gambar + angka.
2. **Hipotesis** — penyebab yang diduga.
3. **Eksperimen** — apa yang dibandingkan (kelompok, split, model) dan kenapa itu menguji hipotesis.
4. **Hasil** — tabel/grafik dari notebook ini.
5. **Kesimpulan** — hipotesis didukung atau tidak, dan saran perbaikan.

In [ ]:
print("File di", WORK)
for p in sorted(WORK.glob("*.csv")):
    print("  ", p.name)
print("Gambar di", FIG_DIR)
for p in sorted(FIG_DIR.glob("*.png")):
    print("  ", p.name)